# Module 2 — Topic 5: Descriptive Statistics
## Live Demo Notebook

**Scenario:** The CEO reviewed the transformation report from Topic 4 and now wants to go deeper. She asks:

> *"I see the numbers — but what do they really mean? What is the typical order? How spread out are our revenues? Are there any unusual transactions we should investigate? And is there a relationship between how much customers spend and how many items they buy?"*

We'll answer each question using descriptive statistics on `orders_clean.csv`.

---

## Part 1 — Load Data and First Statistical Look

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../../data/orders_clean.csv", encoding="utf-8-sig",
                 parse_dates=["order_date"])

print(f"Loaded: {df.shape[0]} rows x {df.shape[1]} columns")
df[["order_id", "product_category", "quantity", "unit_price",
    "total_amount", "state"]].head(5)

In [ ]:
# First statistical snapshot — all numeric columns
df.describe().round(2)

In [ ]:
# Apply the describe() checklist to total_amount
stats = df["total_amount"].describe()

print("===== total_amount — Describe Checklist =====")
print(f"1. count:        {stats['count']:.0f} (expected 490 — any missing?)")
print(f"2. mean:         NGN {stats['mean']:>10,.0f}")
print(f"   median (50%): NGN {stats['50%']:>10,.0f}  ← gap = NGN {stats['mean'] - stats['50%']:,.0f}")
print(f"3. std/mean:     {stats['std']/stats['mean']:.2f}       ← >1 signals heavy spread")
print(f"4. min:          NGN {stats['min']:>10,.0f}  ← sensible?")
print(f"   max:          NGN {stats['max']:>10,.0f}  ← sensible?")
print(f"5. IQR (Q3-Q1):  NGN {stats['75%']-stats['25%']:>10,.0f}  ← middle 50% spread")

---
## Part 2 — Central Tendency: Mean, Median, Mode

In [ ]:
amounts = df["total_amount"]

mean_val   = amounts.mean()
median_val = amounts.median()
mode_val   = amounts.mode()[0]

print("===== Central Tendency: total_amount =====")
print(f"Mean:   NGN {mean_val:>10,.2f}  ← arithmetic average")
print(f"Median: NGN {median_val:>10,.2f}  ← middle value (50th percentile)")
print(f"Mode:   NGN {mode_val:>10,.2f}  ← most frequent value")
print()
print(f"Mean - Median gap: NGN {mean_val - median_val:,.2f}")
print(f"Direction: Mean > Median → RIGHT-SKEWED distribution")
print(f"Interpretation: A small number of high-value orders are")
print(f"                pulling the average significantly above the typical order.")

In [ ]:
# Mode is most useful for categorical columns
print("Most common product category:", df["product_category"].mode()[0])
print("Most common state:",            df["state"].mode()[0])
print("Most common payment method:",   df["payment_method"].mode()[0])
print("Most common delivery status:",  df["delivery_status"].mode()[0])
print("Most common quantity ordered:", df["quantity"].mode()[0])

---
## Part 3 — Spread: Range, Std Dev, IQR

In [ ]:
print("===== Spread Measures: total_amount =====")
print()

# Range
data_range = amounts.max() - amounts.min()
print(f"Range:        NGN {data_range:>10,.0f}  (max - min)")
print(f"  Min:        NGN {amounts.min():>10,.0f}")
print(f"  Max:        NGN {amounts.max():>10,.0f}")
print()

# Standard deviation
std_val = amounts.std()
print(f"Std Dev:      NGN {std_val:>10,.0f}")
print(f"Interpretation: On average, orders deviate NGN {std_val:,.0f}")
print(f"               from the mean of NGN {mean_val:,.0f}")
print()

# IQR
q1  = amounts.quantile(0.25)
q2  = amounts.quantile(0.50)
q3  = amounts.quantile(0.75)
iqr = q3 - q1
print(f"Q1 (25th pct):  NGN {q1:>8,.0f}  — 25% of orders are below this")
print(f"Q2 (50th pct):  NGN {q2:>8,.0f}  — median")
print(f"Q3 (75th pct):  NGN {q3:>8,.0f}  — 75% of orders are below this")
print(f"IQR:            NGN {iqr:>8,.0f}  — spread of the middle 50%")

In [ ]:
# Full percentile breakdown
percentiles = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
print("Percentile breakdown of total_amount:")
for p in percentiles:
    val = amounts.quantile(p)
    print(f"  {p*100:>4.0f}th percentile: NGN {val:>10,.0f}")

---
## Part 4 — Distribution Shape and Skewness

In [ ]:
skewness = amounts.skew()
kurtosis = amounts.kurtosis()

print("===== Distribution Shape =====")
print(f"Skewness: {skewness:.3f}")
if skewness > 1:
    shape = "Heavily right-skewed (long right tail)"
elif skewness > 0.5:
    shape = "Moderately right-skewed"
elif skewness > -0.5:
    shape = "Approximately symmetric"
elif skewness > -1:
    shape = "Moderately left-skewed"
else:
    shape = "Heavily left-skewed (long left tail)"
print(f"Shape:    {shape}")
print()
print(f"Kurtosis: {kurtosis:.3f}")
print(f"Interpretation: {'Heavy tails — more extreme values than normal' if kurtosis > 0 else 'Light tails'} distribution")
print()
print("Key signal: Mean >> Median confirms right skew")
print(f"  Mean ({mean_val:,.0f}) - Median ({median_val:,.0f}) = {mean_val-median_val:,.0f}")

In [ ]:
# Compare skewness across numeric columns
numeric_cols = ["quantity", "unit_price", "total_amount"]
print("Skewness by column:")
for col in numeric_cols:
    sk = df[col].skew()
    label = "right-skewed" if sk > 0.5 else ("left-skewed" if sk < -0.5 else "symmetric")
    print(f"  {col:<16}: {sk:>6.3f}  ({label})")

---
## Part 5 — Outlier Detection: IQR Method

In [ ]:
# IQR fences
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr

print("===== IQR Outlier Detection =====")
print(f"Q1:           NGN {q1:>10,.0f}")
print(f"Q3:           NGN {q3:>10,.0f}")
print(f"IQR:          NGN {iqr:>10,.0f}")
print(f"Lower fence:  NGN {lower_fence:>10,.0f}  (Q1 - 1.5 x IQR)")
print(f"Upper fence:  NGN {upper_fence:>10,.0f}  (Q3 + 1.5 x IQR)")

In [ ]:
# Flag all outliers
outlier_mask = (
    (df["total_amount"] < lower_fence) |
    (df["total_amount"] > upper_fence)
)
outliers = df[outlier_mask].copy()

print(f"Outliers detected: {len(outliers)} orders ({len(outliers)/len(df)*100:.1f}% of dataset)")
print()
print(outliers[["order_id", "customer_name", "state", "product_category",
                "quantity", "unit_price", "total_amount"]]
      .sort_values("total_amount", ascending=False)
      .to_string(index=False))

In [ ]:
# Investigate: are these outliers plausible?
print("===== Outlier Investigation =====")
print()
print("Extreme outliers (above NGN 200,000):")
extreme = outliers[outliers["total_amount"] > 200000]
for _, row in extreme.iterrows():
    implied_unit = row["total_amount"] / row["quantity"]
    print(f"  {row['order_id']}: {row['product_category']}, qty={row['quantity']}, "
          f"total=NGN {row['total_amount']:,.0f}, "
          f"implied unit price=NGN {implied_unit:,.0f}")
    print(f"  Verdict: {'Plausible bulk Electronics purchase' if implied_unit < 100000 else 'Investigate — very high unit price'}")
    print()

print("Decision: Retain all outliers — values are consistent with legitimate large orders.")
print("Using MEDIAN (not mean) as the central tendency measure for revenue reporting.")

---
## Part 6 — Outlier Detection: Z-Score Method

In [ ]:
# Z-score for total_amount
df["z_score"] = (
    (df["total_amount"] - df["total_amount"].mean()) /
     df["total_amount"].std()
).round(3)

extreme_z = df[df["z_score"].abs() > 3]

print(f"Extreme outliers by z-score (|z| > 3): {len(extreme_z)}")
print()
print(extreme_z[["order_id", "total_amount", "z_score"]]
      .sort_values("z_score", ascending=False)
      .to_string(index=False))

In [ ]:
# Compare: IQR vs z-score — do they agree?
iqr_ids    = set(outliers["order_id"])
zscore_ids = set(extreme_z["order_id"])

print(f"Flagged by IQR method:     {len(iqr_ids)} orders")
print(f"Flagged by z-score method: {len(zscore_ids)} orders")
print(f"Flagged by both:           {len(iqr_ids & zscore_ids)} orders")
print(f"Flagged by IQR only:       {len(iqr_ids - zscore_ids)} orders")
print(f"Flagged by z-score only:   {len(zscore_ids - iqr_ids)} orders")
print()
print("Why the difference?")
print("IQR is robust to outliers — its bounds don't change when extremes are present.")
print("Z-score uses mean and std — both inflated by the extreme outliers themselves,")
print("making the z-score method LESS sensitive for right-skewed data.")
print("Conclusion: Use IQR for this dataset.")

---
## Part 7 — Correlation

In [ ]:
# Pairwise correlations
numeric_cols = ["quantity", "unit_price", "total_amount"]

print("===== Correlation Matrix =====")
corr = df[numeric_cols].corr().round(3)
print(corr)
print()

# Interpret each pair
r_qty_amount   = df["quantity"].corr(df["total_amount"])
r_price_amount = df["unit_price"].corr(df["total_amount"])
r_qty_price    = df["quantity"].corr(df["unit_price"])

print(f"quantity vs total_amount:  r = {r_qty_amount:.3f}")
print(f"unit_price vs total_amount: r = {r_price_amount:.3f}")
print(f"quantity vs unit_price:     r = {r_qty_price:.3f}")

In [ ]:
# Interpret correlations in business terms
def interpret_r(r):
    abs_r = abs(r)
    direction = "positive" if r > 0 else "negative"
    if abs_r >= 0.9: strength = "very strong"
    elif abs_r >= 0.7: strength = "strong"
    elif abs_r >= 0.4: strength = "moderate"
    elif abs_r >= 0.1: strength = "weak"
    else: strength = "negligible"
    return f"{strength} {direction} (r = {r:.3f})"

print("Business interpretation:")
print(f"  quantity ↔ total_amount:   {interpret_r(r_qty_amount)}")
print(f"  → Higher quantity orders tend to have higher total amounts")
print()
print(f"  unit_price ↔ total_amount: {interpret_r(r_price_amount)}")
print(f"  → More expensive items are associated with higher total order values")
print()
print(f"  quantity ↔ unit_price:     {interpret_r(r_qty_price)}")
print(f"  → Little relationship between how many items and how expensive they are")

---
## Part 8 — Comparing Groups Statistically

In [ ]:
# Statistical summary per product category
print("===== total_amount Statistics by Category =====")
category_stats = (
    df.groupby("product_category")["total_amount"]
      .agg(
          count="count",
          mean="mean",
          median="median",
          std="std",
          q75=lambda x: x.quantile(0.75)
      )
      .round(0)
      .sort_values("median", ascending=False)
)
print(category_stats)

In [ ]:
# State comparison — median is the right measure for skewed data
print("===== Median Order Value by State =====")
state_medians = (
    df.groupby("state")["total_amount"]
      .agg(orders="count", median="median", mean="mean")
      .round(0)
      .sort_values("median", ascending=False)
)
print(state_medians)
print()
print("Note: Median used (not mean) because total_amount is right-skewed.")
print("Mean would be distorted by high-value outliers in some states.")

In [ ]:
# Are higher-value orders more or less likely to be delivered?
print("===== Order Value by Delivery Status =====")
delivery_stats = (
    df.groupby("delivery_status")["total_amount"]
      .agg(count="count", median="median", mean="mean")
      .round(0)
      .sort_values("median", ascending=False)
)
print(delivery_stats)
print()
print("Business question: Do higher-value orders have different delivery rates?")
delivered_median    = df[df["delivery_status"]=="Delivered"]["total_amount"].median()
not_delivered_median = df[df["delivery_status"]!="Delivered"]["total_amount"].median()
print(f"  Delivered orders median:     NGN {delivered_median:,.0f}")
print(f"  Non-delivered orders median: NGN {not_delivered_median:,.0f}")

---
## Part 9 — The Statistical Report

Answering the CEO's four questions with evidence.

In [ ]:
print("=" * 60)
print("        STATISTICAL ANALYSIS REPORT")
print("=" * 60)

print("\nQ1: What is the TYPICAL order value?")
print(f"   Median: NGN {amounts.median():,.0f}  ← the best measure for skewed data")
print(f"   (Mean is NGN {amounts.mean():,.0f} — inflated by outliers)")

print("\nQ2: How SPREAD OUT are our revenues?")
print(f"   IQR: NGN {iqr:,.0f}")
print(f"   Middle 50% of orders: NGN {q1:,.0f} to NGN {q3:,.0f}")
print(f"   Standard deviation: NGN {amounts.std():,.0f}")
print(f"   Distribution: Right-skewed (skewness = {amounts.skew():.1f})")

print("\nQ3: Any UNUSUAL transactions to investigate?")
print(f"   {len(outliers)} orders fall outside the IQR fence (NGN {upper_fence:,.0f} upper bound)")
print(f"   Top 2 by value: ORD-0011 (NGN 285,000) and ORD-0056 (NGN 312,000)")
print(f"   Verdict: Both are plausible bulk Electronics orders — RETAINED")

print("\nQ4: Relationship between quantity and spend?")
print(f"   Correlation (quantity vs total_amount): r = {r_qty_amount:.3f}")
print(f"   Interpretation: {interpret_r(r_qty_amount)}")
print(f"   Customers ordering more items tend to spend more — expected.")
print(f"   The relationship is not perfect because unit prices vary widely.")

print("\n" + "=" * 60)

---
## Summary

In this demo we:
- Applied the `describe()` checklist to identify key statistical properties
- Calculated and interpreted mean, median, and mode for both numeric and categorical columns
- Measured spread: range, std deviation, IQR, and percentile breakdown
- Quantified skewness and confirmed the right-skewed shape of `total_amount`
- Detected outliers using the IQR method and investigated their plausibility
- Compared IQR vs z-score and explained why IQR is better for skewed data
- Calculated the correlation matrix and interpreted each pair in business terms
- Compared groups statistically — categories, states, delivery status
- Produced a structured statistical report answering four business questions

**Next — Topic 6:** Data Privacy & Protection. Now that we can extract powerful insights from data, we'll learn our legal and ethical obligations to the people whose data we're using — and practice anonymising PII before sharing any analysis.